# Bias-Varianz-Tradeoff & Kreuzvalidierung

Dies ist eines der fundamentalsten Konzepte im Machine Learning.  
Es erklärt **warum Modelle versagen** und wie man es vermeidet.

## Inhaltsverzeichnis
1. Das Bias-Varianz-Dilemma
2. Underfitting vs. Overfitting visualisiert
3. Modellauswahl mit einfacher Validierung
4. Cross-Validation (K-Fold)
5. Regularisierung als Lösung


## 1. Das Bias-Varianz-Dilemma

### Bias (Verzerrung)
- **Hoher Bias:** Das Modell ist zu **vereinfacht** — es kann die wahren Muster nicht lernen
- Sowohl Trainings- als auch Testfehler sind hoch
- Beispiel: Lineare Regression für eine stark kurvenförmige Beziehung

### Varianz
- **Hohe Varianz:** Das Modell ist zu **komplex** — es lernt auch das Rauschen in den Trainingsdaten
- Trainingsfehler niedrig, Testfehler viel höher
- Beispiel: Decision Tree ohne Tiefenbeschränkung

### Das Dilemma
```
                Komplex werden
    Bias ←─────────────────────── Bias ↓
    ↑                                   ↓
    Einfaches Modell         Komplexes Modell
    ↓                                   ↑
    Varianz ↓ ──────────────────────→ Varianz
                Einfach bleiben
```

**Ziel:** Den "Sweet Spot" in der Mitte finden — gute Generalisierung!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Daten generieren: y = sin(x) + Rauschen
np.random.seed(42)
X = np.sort(np.random.uniform(0, 2*np.pi, 50))
y = np.sin(X) + np.random.normal(0, 0.3, 50)

X = X.reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titel = ['Hoher Bias (Underfitting)
Grad 1', 
         'Gutes Gleichgewicht
Grad 3', 
         'Hohe Varianz (Overfitting)
Grad 12']
grade = [1, 3, 12]
farben = ['red', 'green', 'purple']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

for ax, grad, titel_str, farbe in zip(axes, grade, titel, farben):
    poly = PolynomialFeatures(degree=grad)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)
    
    modell = LinearRegression().fit(X_tr_p, y_tr)
    
    X_plot = np.linspace(0, 2*np.pi, 300).reshape(-1, 1)
    y_plot = modell.predict(poly.transform(X_plot))
    
    train_rmse = np.sqrt(mean_squared_error(y_tr, modell.predict(X_tr_p)))
    test_rmse  = np.sqrt(mean_squared_error(y_te, modell.predict(X_te_p)))
    
    ax.scatter(X_tr, y_tr, alpha=0.7, s=20, label='Training')
    ax.scatter(X_te, y_te, alpha=0.7, s=20, marker='s', label='Test')
    ax.plot(X_plot, y_plot, color=farbe, lw=2)
    ax.set_title(f"{titel_str}\nTrain RMSE: {train_rmse:.2f} | Test RMSE: {test_rmse:.2f}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Bias-Varianz-Tradeoff visualisiert", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Diagnose: Bin ich beim Overfitting oder Underfitting?

### Lernkurven als Diagnose-Werkzeug

Trainiere das Modell mit verschieden großen Trainingsmengen und beobachte wie sich Train- und Validierungsfehler entwickeln.

**Overfitting:**
- Trainings-Score: hoch
- Validierungs-Score: niedrig
- Großer Unterschied zwischen beiden

**Underfitting:**
- Trainings-Score: niedrig
- Validierungs-Score: niedrig
- Kein großer Unterschied (beide schlecht)


In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tiefe, titel in zip(axes, [2, 20], ['Underfitting (max_depth=2)', 'Overfitting (max_depth=20)']):
    modell = DecisionTreeClassifier(max_depth=tiefe, random_state=42)
    
    train_sizes, train_scores, val_scores = learning_curve(
        modell, X_c, y_c, cv=5, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy'
    )
    
    ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training', color='blue')
    ax.fill_between(train_sizes, 
                    train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1),
                    alpha=0.1, color='blue')
    ax.plot(train_sizes, val_scores.mean(axis=1), 's-', label='Validierung', color='orange')
    ax.fill_between(train_sizes,
                    val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1),
                    alpha=0.1, color='orange')
    
    ax.set_xlabel("Trainingsgröße")
    ax.set_ylabel("Genauigkeit")
    ax.set_title(titel)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.7, 1.05])

plt.tight_layout()
plt.show()

## 3. Einfache Validierung: Train / Validation / Test

**Drei Datensätze statt zwei:**
- **Training** (60%): Modell lernt daraus
- **Validierung** (20%): Hyperparameter-Tuning, Modellauswahl
- **Test** (20%): Finale Bewertung — EINMAL am Ende!

**Warum drei?**  
Wenn wir Hyperparameter anhand der Testdaten optimieren, "schummeln" wir — die Testdaten sind nicht mehr wirklich "ungesehen".

**Analogie:** 
- Training = Lernunterlagen
- Validierung = Probeprüfung (darf man wiederholen)
- Test = echte Abschlussprüfung (einmal, am Ende)


In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import pandas as pd

# Autodaten laden
auto = fetch_openml(name='autos', version=1, as_frame=True)
df = auto.frame.dropna()

# Numerische Features und Ziel
ziel = 'price'
features = df.select_dtypes(include=[np.number]).columns.drop(ziel, errors='ignore').tolist()
X_auto = df[features].dropna()
y_auto = pd.to_numeric(df.loc[X_auto.index, ziel], errors='coerce').dropna()
X_auto = X_auto.loc[y_auto.index]

print(f"Datenpunkte: {X_auto.shape[0]}, Features: {X_auto.shape[1]}")

# 60/20/20 Split
X_temp, X_te, y_temp, y_te = train_test_split(X_auto, y_auto, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Training: {len(X_tr)} | Validierung: {len(X_val)} | Test: {len(X_te)}")

# Drei Modelle vergleichen
scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_te_s  = scaler.transform(X_te)

modelle = {
    'Lineare Regression': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1),
    'Ridge (α=100)': Ridge(alpha=100),
}

print(f"\n{'Modell':<25} {'Val R²':>10}")
print("-" * 38)
beste_val = -np.inf
bestes_modell = None
for name, m in modelle.items():
    m.fit(X_tr_s, y_tr)
    val_score = r2_score(y_val, m.predict(X_val_s))
    print(f"{name:<25} {val_score:>10.4f}")
    if val_score > beste_val:
        beste_val = val_score
        bestes_modell = (name, m)

print(f"\n→ Bestes Modell: {bestes_modell[0]}")
finale = r2_score(y_te, bestes_modell[1].predict(X_te_s))
print(f"→ Finale Test-Genauigkeit: {finale:.4f}")

## 4. Cross-Validation (K-Fold Kreuzvalidierung)

**Problem mit einfacher Validierung:** Das Ergebnis hängt vom zufälligen Split ab — andere `random_state` → andere Antwort!

**Lösung: K-Fold Cross-Validation**

1. Daten in k gleichgroße Teile ("Folds") aufteilen
2. k Mal trainieren:
   - Fold i = Validierungsset
   - Alle anderen Folds = Trainingsset
3. k Ergebnisse mitteln

**Vorteil:** Jeder Datenpunkt wird genau einmal als Validation verwendet → zuverlässigeres Ergebnis!

**Typischer Wert:** k = 5 oder k = 10


In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

cancer = load_breast_cancer()
X_cv, y_cv = cancer.data, cancer.target

# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

modelle_cv = {
    'Logistische Regression': LogisticRegression(max_iter=10000),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'KNN (k=15)': KNeighborsClassifier(n_neighbors=15),
}

print("5-Fold Cross-Validation:")
print(f"{'Modell':<30} {'Mean Accuracy':>15} {'Std Dev':>10}")
print("-" * 58)
for name, modell in modelle_cv.items():
    scores = cross_val_score(modell, X_cv, y_cv, cv=kf, scoring='accuracy')
    print(f"{name:<30} {scores.mean():>15.4f} {scores.std():>10.4f}")
    print(f"{'':30} {' '.join([f'{s:.3f}' for s in scores])}")
    print()

In [ ]:
# Warum CV zuverlässiger ist: Zeige Stabilität
np.random.seed(0)

print("Vergleich: Einfacher Split vs. 5-Fold CV bei verschiedenen Random States:\n")
from sklearn.linear_model import LogisticRegression

modell = LogisticRegression(max_iter=10000)

print(f"{'Random State':>13} {'Einfacher Split':>17} {'5-Fold CV (stabil)':>20}")
print("-" * 53)

for rs in [0, 1, 2, 3, 4]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_cv, y_cv, test_size=0.2, random_state=rs)
    modell.fit(X_tr, y_tr)
    einzel_score = modell.score(X_te, y_te)
    
    cv_scores = cross_val_score(modell, X_cv, y_cv, cv=5)
    
    print(f"{rs:>13} {einzel_score:>17.4f} {cv_scores.mean():>20.4f}")

print("\n→ CV-Ergebnis ist konsistenter (stabiler)!")

## 5. Regularisierung als Lösung für Overfitting

Wir haben Ridge und Lasso schon gesehen — jetzt verstehen wir **warum** sie helfen:

**Ohne Regularisierung:** Modell kann beliebig komplexe Koeffizienten lernen → Overfitting
**Mit Regularisierung:** Koeffizienten werden "in Zaum gehalten" → weniger Overfitting

### Hyperparameter-Tuning mit Cross-Validation

Das ist der richtige Weg, um das beste α für Ridge/Lasso zu finden!


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

dataset = fetch_openml(name='boston', version=1)
X_bos = dataset.data.astype(float)
y_bos = dataset.target.astype(float)

scaler = StandardScaler()
X_skaliert = scaler.fit_transform(X_bos)

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
cv_scores = []

for a in alphas:
    ridge = Ridge(alpha=a)
    scores = cross_val_score(ridge, X_skaliert, y_bos, cv=5, scoring='r2')
    cv_scores.append(scores.mean())

plt.figure(figsize=(8, 5))
plt.semilogx(alphas, cv_scores, 'o-', color='steelblue', markersize=6)
plt.xlabel("Alpha (log-Skala)")
plt.ylabel("CV R²-Score")
plt.title("Ridge: Optimales Alpha per Cross-Validation")
plt.grid(True, alpha=0.3)
plt.axvline(alphas[np.argmax(cv_scores)], color='red', linestyle='--', 
            label=f"Bestes α: {alphas[np.argmax(cv_scores)]}")
plt.legend()
plt.show()

print(f"Bestes Alpha: {alphas[np.argmax(cv_scores)]}")
print(f"Bester CV-Score: {max(cv_scores):.4f}")

## Zusammenfassung

| Problem | Diagnose | Lösung |
|---------|---------|--------|
| **Underfitting** | Trainings- UND Testfehler hoch | Komplexeres Modell, mehr Features, weniger Regularisierung |
| **Overfitting** | Trainingsfehler niedrig, Testfehler hoch | Weniger Features, mehr Daten, Regularisierung, kleineres Modell |
| **Instabile Ergebnisse** | Stark wechselnde Scores | Cross-Validation statt einfachem Split |

**Goldene Regeln:**
1. Immer Train/Test trennen — Testdaten sind heilig!
2. Hyperparameter auf Validierungsdaten tunen, nicht Testdaten
3. CV gibt zuverlässigere Schätzungen als einfacher Split
4. Lernkurven zeigen ob du Overfitting oder Underfitting hast
